In [7]:
import jax.numpy as jnp
import jax
from scenarios.charger_topologies import create_uniform_topology
from scenarios.ev_arivals import interpolate_arrival_data
from util.helpers import pretty_print_charger_group
from environment import CarState, ChargerGroup, ChargerState
from scenarios.scenario_data import office_distribution_means
import chex
import equinox as eqx

grid_connection = create_uniform_topology(3, 1)
arrival_distributions = interpolate_arrival_data(
    list(office_distribution_means.values()), 5, 1
)
pretty_print_charger_group(grid_connection)

└── ChargerGroup(group_capacity_max=150.0, group_capacity_current=0.0)
    ├── ChargerGroup(group_capacity_max=50.0, group_capacity_current=0.0)
    │   └── ChargerState(charge_rate_current=0.0, car_connected=False)
    │       └── No Car Connected
    ├── ChargerGroup(group_capacity_max=50.0, group_capacity_current=0.0)
    │   └── ChargerState(charge_rate_current=0.0, car_connected=False)
    │       └── No Car Connected
    └── ChargerGroup(group_capacity_max=50.0, group_capacity_current=0.0)
        └── ChargerState(charge_rate_current=0.0, car_connected=False)
            └── No Car Connected


In [8]:
def set_a_car(connection, index):
    connection = eqx.tree_at(
        lambda x: x.connections[index].connections[0].car, 
        connection, 
        CarState(
            time_till_leave=10,
            battery_level=50.,
            battery_capacity=250.,
            battery_temperature=35.0,
            car_rate_max=75.0
        )
    )
    connection = eqx.tree_at(
        lambda x: x.connections[index].connections[0].car_connected, 
        connection, 
        True
    )
    return connection
# grid_connection = set_a_car(grid_connection, 0)
grid_connection = set_a_car(grid_connection, 0)
# grid_connection = set_a_car(grid_connection, -1)
pretty_print_charger_group(grid_connection)

└── ChargerGroup(group_capacity_max=150.0, group_capacity_current=0.0)
    ├── ChargerGroup(group_capacity_max=50.0, group_capacity_current=0.0)
    │   └── ChargerState(charge_rate_current=0.0, car_connected=True)
    │       └── CarState(time_waiting=0, time_charging=0, time_till_leave=10, battery_level=50.0, battery_capacity=250.0, battery_temperature=35.0)
    ├── ChargerGroup(group_capacity_max=50.0, group_capacity_current=0.0)
    │   └── ChargerState(charge_rate_current=0.0, car_connected=False)
    │       └── No Car Connected
    └── ChargerGroup(group_capacity_max=50.0, group_capacity_current=0.0)
        └── ChargerState(charge_rate_current=0.0, car_connected=False)
            └── No Car Connected


In [3]:
leaves, treedef = jax.tree.flatten(grid_connection, is_leaf=lambda x: isinstance(x, ChargerState))
leaves, treedef

([ChargerState(charger_rate_current=0.0, car_connected=False, car=CarState(car_rate_max=0.0, time_waiting=0.0, time_charging=0.0, time_till_leave=0.0, battery_level=0.0, battery_capacity=0.0, battery_temperature=0.0)),
  ChargerState(charger_rate_current=0.0, car_connected=True, car=CarState(car_rate_max=75.0, time_waiting=0.0, time_charging=0.0, time_till_leave=10, battery_level=50, battery_capacity=250, battery_temperature=35.0)),
  ChargerState(charger_rate_current=0.0, car_connected=False, car=CarState(car_rate_max=0.0, time_waiting=0.0, time_charging=0.0, time_till_leave=0.0, battery_level=0.0, battery_capacity=0.0, battery_temperature=0.0))],
 PyTreeDef(CustomNode(ChargerGroup[('connections',), ('group_capacity_max',), (150.0,)], [[CustomNode(ChargerGroup[('connections',), ('group_capacity_max',), (50.0,)], [[*]]), CustomNode(ChargerGroup[('connections',), ('group_capacity_max',), (50.0,)], [[*]]), CustomNode(ChargerGroup[('connections',), ('group_capacity_max',), (50.0,)], [[*]]

In [44]:
# from the list of ChargerStates (leaves), get the indices of the ones that have a car connected in a jittable way
@jax.jit
def get_connected_indices(leaves):
    connects = jax.tree.map(lambda x: x.car_connected, leaves, is_leaf=lambda x: isinstance(x, ChargerState))
    # now get the indices of the connected chargers
    indices = jnp.nonzero(jnp.array(connects), size=10, fill_value=-1)
    
    return indices

connected_indices = get_connected_indices(leaves)
connected_indices

(Array([ 0,  6,  8, -1, -1, -1, -1, -1, -1, -1], dtype=int32),)

In [4]:
with_cars, without_cars = eqx.partition(
    grid_connection, lambda x: isinstance(x, ChargerState) and x.car_connected, is_leaf=lambda x: isinstance(x, ChargerState)
)
with_cars

ChargerGroup(
  group_capacity_max=150.0,
  connections=[
    ChargerGroup(group_capacity_max=50.0, connections=[None]),
    ChargerGroup(
      group_capacity_max=50.0,
      connections=[
        ChargerState(
          charger_rate_current=0.0,
          car_connected=True,
          car=CarState(
            car_rate_max=75.0,
            time_waiting=0.0,
            time_charging=0.0,
            time_till_leave=10,
            battery_level=50,
            battery_capacity=250,
            battery_temperature=35.0
          )
        )
      ]
    ),
    ChargerGroup(group_capacity_max=50.0, connections=[None])
  ]
)

In [12]:
# @jax.jit
def add_new_cars(key: chex.PRNGKey):
    def sample_car(key: chex.PRNGKey) -> CarState:
        keys = jax.random.split(key, 5)
        departure_time = jax.random.randint(keys[0], (), 10, 60)
        arrival_battery_level = jax.random.uniform(keys[1], (), minval=3., maxval=50.)
        car_battery_capacity = jax.random.uniform(keys[2], (), minval=200., maxval=300.)
        return CarState(
            time_till_leave=departure_time,
            battery_level=arrival_battery_level,
            battery_capacity=car_battery_capacity,
            battery_temperature=35.0,
            car_rate_max=75.0
        )

    new_cars_amount = arrival_distributions[4].sample(seed=key)
    new_cars_amount = jnp.maximum(new_cars_amount, 1).astype(int)




    all_connections = jax.tree.leaves(grid_connection, is_leaf=lambda x: isinstance(x, ChargerState))
    not_connects = jax.tree.map(lambda x: not x.car_connected, all_connections, is_leaf=lambda x: isinstance(x, ChargerState))
    print("not connected:", not_connects)
    not_connects = jnp.array(not_connects)

    # sort the not_connects, but store the ordering so we can restore it later
    sort_order = jnp.argsort(not_connects, descending=True)
    not_connects = not_connects[sort_order]
    print(not_connects)
    # now we remove everything after new_cars_amount (set to FALSE)
    mask = (jnp.arange(grid_connection.number_of_chargers_in_group) < new_cars_amount).astype(bool)
    print("mask", mask)
    # not_connects = not_connects.at[new_cars_amount:].set(False)
    not_connects = not_connects * mask
    print(not_connects)
    # now we restore the original order
    not_connects = not_connects.at[sort_order].set(not_connects)
    not_connects = [n for n in not_connects]


    
    # We cannot sample the right amount of cars (in a vmap)
    # so we always sample the maximum amount, and use what we need.
    new_car_keys = jax.random.split(key, grid_connection.number_of_chargers_in_group)
    # current_chargers = jax.tree_leaves(grid_connection, is_leaf=lambda x: isinstance(x, ChargerState))
    new_chargers = [
        ChargerState(
            charger_rate_current=charger.charger_rate_current,
            car_connected=True,
            car=sample_car(new_car_keys[i])
        ) for i, charger in enumerate(grid_connection.chargers_in_group)
    ]
    print("new_chargers", new_chargers)
    print(len(new_chargers))


    # New chargers is now a list of objects, where each objects contains the property: "car_connected"
    # not_connects is a list of the same length as new_chargers, where each element is a boolean
    # we need to match these two
    # we can do this by setting the car_connected property of the new_chargers to the not_connects
    new_chargers = jax.tree.map(
        lambda x, y: ChargerState(
            charger_rate_current=x.charger_rate_current,
            car_connected=y,
            car=x.car
        ),
        new_chargers,
        not_connects,
        is_leaf=lambda x: isinstance(x, ChargerState)
    )
    print("new_chargers", new_chargers)
    grid_connection_leaves = jax.tree_leaves(grid_connection, is_leaf=lambda x: isinstance(x, ChargerState))
    updated_chargers = jax.tree_map(
        lambda x, y: jax.lax.cond(
            y.car_connected, lambda: y, lambda: x
        ), new_chargers, grid_connection_leaves, is_leaf=lambda x: isinstance(x, ChargerState)
    )

    print("updated_chargers", updated_chargers)


    # with_cars, without_cars = eqx.partition(
    #     grid_connection, lambda x: isinstance(x, ChargerState) and x.car_connected, is_leaf=lambda x: isinstance(x, ChargerState)
    # )

    # _, treedef = jax.tree_flatten(grid_connection, is_leaf=lambda x: isinstance(x, ChargerState))
    # # print(_)
    # new_topology = jax.tree_unflatten(treedef, new_chargers)
    # # print(new_topology)

    # print(eqx.combine(with_cars, new_topology))

    # num_overfilled = grid_connection.number_of_chargers_in_group - len(jax.tree.leaves(with_cars, is_leaf=lambda x: isinstance(x, ChargerState))) - new_cars_amount
    # print(num_overfilled, new_cars_amount)

    # # # now get the indices of the NOT connected chargers
    # indices = jnp.nonzero(jnp.array(not_connects), size=grid_connection.number_of_chargers_in_group, fill_value=-1)[0]
    # print(indices)

    # # set all indices of indices[new_cars_amount:] to -1
    # indices = indices.at[new_cars_amount:].set(-1)
    # print(indices)

    # # make a mask out of this
    # mask = indices != -1
    # print(mask)

    # # # get the number of cars already connected
    # # num_connected = grid_connection.number_of_chargers_in_group - jnp.sum(jnp.array(not_connects))
    # # print(num_connected)



    # We have a list of indices of chargers that are not connected

key = jax.random.PRNGKey(0)
add_new_cars(key)   

# keys = jax.random.split(key, 5)
# jax.vmap(add_new_cars)(keys)

not connected: [False, True, True]
[ True  True False]
mask [ True False False]
[ True False False]
new_chargers [ChargerState(
  charger_rate_current=0.0,
  car_connected=True,
  car=CarState(
    car_rate_max=75.0,
    time_waiting=0,
    time_charging=0,
    time_till_leave=i32[],
    battery_level=f32[],
    battery_capacity=f32[],
    battery_temperature=35.0
  )
), ChargerState(
  charger_rate_current=0.0,
  car_connected=True,
  car=CarState(
    car_rate_max=75.0,
    time_waiting=0,
    time_charging=0,
    time_till_leave=i32[],
    battery_level=f32[],
    battery_capacity=f32[],
    battery_temperature=35.0
  )
), ChargerState(
  charger_rate_current=0.0,
  car_connected=True,
  car=CarState(
    car_rate_max=75.0,
    time_waiting=0,
    time_charging=0,
    time_till_leave=i32[],
    battery_level=f32[],
    battery_capacity=f32[],
    battery_temperature=35.0
  )
)]
3
new_chargers [ChargerState(
  charger_rate_current=0.0,
  car_connected=bool[],
  car=CarState(
    car_